# 🏝️ Sistem Pendukung Keputusan (SPK) - Pemilihan Destinasi Wisata Indonesia

## Metode yang Digunakan:
1. **SMART** (Simple Multi-Attribute Rating Technique)
2. **SAW** (Simple Additive Weighting)
3. **TOPSIS** (Technique for Order Preference by Similarity to Ideal Solution)

## Dataset:
- **Sumber:** [Indonesia Tourism Destination - Kaggle](https://www.kaggle.com/datasets/aprabowo/indonesia-tourism-destination)
- **Kriteria:**
  - `Price` → Harga Tiket Masuk (**Cost** - semakin rendah semakin baik)
  - `Rating` → Rating Pengunjung (**Benefit** - semakin tinggi semakin baik)
  - `Jumlah Review` → Popularitas (**Benefit**)
  - `Time_Minutes` → Estimasi Waktu Kunjungan (**Benefit**)

## 1. Import Library & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

print('✅ Library berhasil di-import!')

In [ ]:
# Load dataset - sesuaikan path jika menggunakan Kaggle
# Jika di Kaggle: /kaggle/input/indonesia-tourism-destination/tourism_with_id.csv
try:
    df_raw = pd.read_csv('/kaggle/input/indonesia-tourism-destination/tourism_with_id.csv')
    df_user = pd.read_csv('/kaggle/input/indonesia-tourism-destination/user.csv')
    df_rating = pd.read_csv('/kaggle/input/indonesia-tourism-destination/tourism_rating.csv')
    print('✅ Dataset berhasil dimuat dari Kaggle!')
except:
    # Fallback: generate sample data yang realistis
    print('⚠️  File tidak ditemukan, menggunakan sample data realistis...')
    np.random.seed(42)
    cities = ['Jakarta', 'Bandung', 'Yogyakarta', 'Bali', 'Surabaya']
    place_templates = {
        'Jakarta': ['Monas','Kota Tua','Ancol','TMII','Ragunan','Kepulauan Seribu','Museum Nasional','Taman Mini'],
        'Bandung': ['Kawah Putih','Tangkuban Perahu','Trans Studio','Lembang','Dusun Bambu','Situ Patenggang','Farmhouse','Orchid Forest'],
        'Yogyakarta': ['Prambanan','Borobudur','Keraton','Malioboro','Parangtritis','Goa Jomblang','Kalibiru','Merapi'],
        'Bali': ['Tanah Lot','Ubud Palace','Kuta Beach','GWK','Seminyak','Uluwatu','Tirta Gangga','Tegallalang'],
        'Surabaya': ['Kebun Binatang','House of Sampoerna','Tugu Pahlawan','Pantai Kenjeran','Galaxy Mall','Ciputra Waterpark','Suramadu','G-Walk']
    }
    records = []
    place_id = 1
    for city in cities:
        for place in place_templates[city]:
            records.append({
                'Place_Id': place_id,
                'Place_Name': place,
                'Category': np.random.choice(['Budaya','Alam','Taman Hiburan','Bahari','Pusat Perbelanjaan']),
                'City': city,
                'Price': np.random.choice([0, 5000, 10000, 15000, 20000, 25000, 30000, 50000, 75000, 100000]),
                'Rating': round(np.random.uniform(3.5, 5.0), 1),
                'Time_Minutes': np.random.choice([60, 90, 120, 150, 180, 240, 300]),
                'Coordinate': f"{np.random.uniform(-8,-6):.4f}, {np.random.uniform(106,115):.4f}",
                'Lat': np.random.uniform(-8, -6),
                'Long': np.random.uniform(106, 115),
                'Description': f'Destinasi wisata populer di {city}'
            })
            place_id += 1
    df_raw = pd.DataFrame(records)
    
    # Buat rating dengan jumlah review acak
    ratings = []
    for pid in df_raw['Place_Id']:
        n_reviews = np.random.randint(50, 500)
        for _ in range(n_reviews):
            ratings.append({'User_Id': np.random.randint(1,300), 'Place_Id': pid, 'Place_Ratings': np.random.randint(1,6)})
    df_rating = pd.DataFrame(ratings)
    print('✅ Sample data berhasil dibuat!')

print(f'\n📊 Shape dataset utama: {df_raw.shape}')
print(f'📊 Shape dataset rating: {df_rating.shape}')
display(df_raw.head())

## 2. Eksplorasi & Preprocessing Data

In [ ]:
print('=== INFO DATASET ===')
df_raw.info()
print('\n=== STATISTIK DESKRIPTIF ===')
display(df_raw.describe())

In [ ]:
print('=== MISSING VALUES ===')
print(df_raw.isnull().sum())

print('\n=== DISTRIBUSI KOTA ===')
print(df_raw['City'].value_counts())

print('\n=== DISTRIBUSI KATEGORI ===')
print(df_raw['Category'].value_counts())

In [ ]:
# Hitung jumlah review per tempat dari df_rating
review_count = df_rating.groupby('Place_Id').size().reset_index(name='Jumlah_Review')

# Merge
df = df_raw.copy()
df = df.merge(review_count, on='Place_Id', how='left')
df['Jumlah_Review'] = df['Jumlah_Review'].fillna(0).astype(int)

# Pilih kolom yang diperlukan
df_clean = df[['Place_Id', 'Place_Name', 'Category', 'City',
               'Price', 'Rating', 'Jumlah_Review', 'Time_Minutes']].copy()

# Bersihkan nilai kosong
df_clean.dropna(subset=['Price', 'Rating', 'Time_Minutes'], inplace=True)
df_clean['Time_Minutes'] = pd.to_numeric(df_clean['Time_Minutes'], errors='coerce').fillna(60)

print(f'✅ Data bersih: {df_clean.shape[0]} destinasi')
display(df_clean.head(10))

In [ ]:
# Visualisasi EDA
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Eksplorasi Data Destinasi Wisata Indonesia', fontsize=15, fontweight='bold')

# 1. Distribusi per Kota
city_counts = df_clean['City'].value_counts()
axes[0,0].bar(city_counts.index, city_counts.values, color=sns.color_palette('Set2'))
axes[0,0].set_title('Jumlah Destinasi per Kota')
axes[0,0].set_xlabel('Kota')
axes[0,0].set_ylabel('Jumlah')
axes[0,0].tick_params(axis='x', rotation=15)

# 2. Distribusi Harga
axes[0,1].hist(df_clean['Price'], bins=20, color='steelblue', edgecolor='white')
axes[0,1].set_title('Distribusi Harga Tiket Masuk')
axes[0,1].set_xlabel('Harga (Rp)')
axes[0,1].set_ylabel('Frekuensi')

# 3. Distribusi Rating
axes[1,0].hist(df_clean['Rating'], bins=15, color='coral', edgecolor='white')
axes[1,0].set_title('Distribusi Rating Pengunjung')
axes[1,0].set_xlabel('Rating')
axes[1,0].set_ylabel('Frekuensi')

# 4. Rating vs Price
colors = sns.color_palette('Set1', len(df_clean['City'].unique()))
city_color = {city: colors[i] for i, city in enumerate(df_clean['City'].unique())}
for city, grp in df_clean.groupby('City'):
    axes[1,1].scatter(grp['Price'], grp['Rating'], label=city,
                      color=city_color[city], alpha=0.7, s=50)
axes[1,1].set_title('Harga vs Rating per Kota')
axes[1,1].set_xlabel('Harga (Rp)')
axes[1,1].set_ylabel('Rating')
axes[1,1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_tourism.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualisasi EDA selesai')

## 3. Definisi Kriteria & Bobot (SMART Framework)

**SMART** digunakan untuk menentukan bobot kriteria secara terstruktur.

| Kriteria | Tipe | Bobot | Justifikasi |
|----------|------|-------|-------------|
| Price | Cost (↓) | 0.30 | Keterjangkauan harga sangat penting bagi wisatawan |
| Rating | Benefit (↑) | 0.35 | Kualitas pengalaman wisata (prioritas utama) |
| Jumlah Review | Benefit (↑) | 0.20 | Popularitas & kepercayaan destinasi |
| Time_Minutes | Benefit (↑) | 0.15 | Nilai pengalaman (lebih lama = lebih berkesan) |

In [ ]:
# ============================================================
#  KONFIGURASI KRITERIA - Ubah di sini jika ingin eksperimen
# ============================================================

CRITERIA = ['Price', 'Rating', 'Jumlah_Review', 'Time_Minutes']
CRITERIA_LABELS = {
    'Price': 'Harga Tiket (Rp)',
    'Rating': 'Rating Pengunjung',
    'Jumlah_Review': 'Jumlah Review',
    'Time_Minutes': 'Waktu Kunjungan (menit)'
}
WEIGHTS = {
    'Price': 0.30,
    'Rating': 0.35,
    'Jumlah_Review': 0.20,
    'Time_Minutes': 0.15
}
BENEFIT = {
    'Price': False,       # Cost  → semakin kecil semakin baik
    'Rating': True,       # Benefit → semakin besar semakin baik
    'Jumlah_Review': True,
    'Time_Minutes': True
}

# Validasi bobot
total_w = sum(WEIGHTS.values())
assert abs(total_w - 1.0) < 1e-9, f'Total bobot harus = 1, sekarang = {total_w}'
print('✅ Konfigurasi kriteria:')
for c in CRITERIA:
    tipe = 'Benefit (↑)' if BENEFIT[c] else 'Cost   (↓)'
    print(f'   {CRITERIA_LABELS[c]:<30} | Tipe: {tipe} | Bobot: {WEIGHTS[c]:.2f}')
print(f'   {"Total Bobot":30} | {total_w:.2f}')

In [ ]:
# Visualisasi bobot dengan pie chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

labels = [CRITERIA_LABELS[c] for c in CRITERIA]
sizes  = [WEIGHTS[c] for c in CRITERIA]
colors_pie = ['#FF6B6B','#4ECDC4','#45B7D1','#FFA07A']
explode = [0.05]*len(CRITERIA)

ax1.pie(sizes, explode=explode, labels=labels, colors=colors_pie,
        autopct='%1.1f%%', shadow=True, startangle=140,
        textprops={'fontsize': 10})
ax1.set_title('Distribusi Bobot Kriteria (SMART)', fontweight='bold')

# Bar chart bobot
bars = ax2.barh(labels, sizes, color=colors_pie, edgecolor='white', height=0.5)
for bar, val in zip(bars, sizes):
    ax2.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
             f'{val:.0%}', va='center', fontweight='bold')
ax2.set_xlim(0, max(sizes)*1.2)
ax2.set_title('Bobot Kriteria SMART', fontweight='bold')
ax2.set_xlabel('Bobot')
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('bobot_smart.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Filtering Data (Opsional: Per Kota)

Untuk efisiensi komputasi, kita bisa memilih subset data (misal top 50 per kota atau filter per kota tertentu).

In [ ]:
# ============================================================
#  FILTER - Ubah SELECTED_CITY = None untuk semua kota
# ============================================================
SELECTED_CITY = None   # contoh: 'Bali' | None = semua kota
TOP_N = 50             # Ambil top N berdasarkan rating tertinggi

if SELECTED_CITY:
    df_filtered = df_clean[df_clean['City'] == SELECTED_CITY].copy()
    print(f'🔍 Filter kota: {SELECTED_CITY} → {len(df_filtered)} destinasi')
else:
    df_filtered = df_clean.copy()
    print(f'🔍 Semua kota → {len(df_filtered)} destinasi')

# Ambil top N
df_filtered = df_filtered.nlargest(TOP_N, 'Rating').reset_index(drop=True)
df_filtered.index += 1

print(f'📋 Dataset akhir untuk SPK: {len(df_filtered)} destinasi')
display(df_filtered[['Place_Name','City','Price','Rating','Jumlah_Review','Time_Minutes']].head(15))

## 5. Metode SAW (Simple Additive Weighting)

**Langkah-langkah SAW:**
1. Buat matriks keputusan X
2. Normalisasi matriks → R (benefit: xij/max, cost: min/xij)
3. Hitung nilai preferensi: V = Σ(wj × rij)
4. Ranking berdasarkan V terbesar

In [ ]:
def saw_method(df, criteria, weights, benefit):
    """
    Simple Additive Weighting (SAW)
    Returns: DataFrame dengan kolom skor SAW dan ranking
    """
    X = df[criteria].values.astype(float)
    R = np.zeros_like(X)
    
    print('📐 Matriks Normalisasi SAW:')
    for j, crit in enumerate(criteria):
        col = X[:, j]
        if benefit[crit]:  # Benefit
            max_val = col.max()
            R[:, j] = col / max_val if max_val != 0 else col
            print(f'   {crit} (Benefit): rij = xij / {max_val:.2f}')
        else:              # Cost
            min_val = col.min()
            # Handle kasus harga = 0 (gratis) dengan pseudo-min
            if min_val == 0:
                # Normalisasi: destinasi gratis dapat nilai 1, lainnya min_nonzero/xij
                nonzero = col[col > 0]
                pseudo_min = nonzero.min() if len(nonzero) > 0 else 1
                R[:, j] = np.where(col == 0, 1.0, pseudo_min / col)
            else:
                R[:, j] = min_val / col
            print(f'   {crit} (Cost):    rij = {min_val:.2f} / xij')
    
    # Hitung skor SAW
    w_arr = np.array([weights[c] for c in criteria])
    scores = R.dot(w_arr)
    
    result = df[['Place_Name', 'City', 'Category'] + criteria].copy()
    result['SAW_Score'] = np.round(scores, 4)
    result['SAW_Rank'] = result['SAW_Score'].rank(ascending=False, method='min').astype(int)
    return result.sort_values('SAW_Rank').reset_index(drop=True), R

df_saw, R_saw = saw_method(df_filtered, CRITERIA, WEIGHTS, BENEFIT)

print('\n🏆 TOP 10 Destinasi - Metode SAW')
print('='*80)
display(df_saw[['SAW_Rank','Place_Name','City','Price','Rating','Jumlah_Review','Time_Minutes','SAW_Score']].head(10))

## 6. Metode TOPSIS

**Langkah-langkah TOPSIS:**
1. Normalisasi matriks (vector normalization)
2. Matriks keputusan terbobot
3. Tentukan Ideal Positif (A+) dan Ideal Negatif (A-)
4. Hitung jarak ke A+ dan A-
5. Hitung nilai preferensi: Ci = Di- / (Di+ + Di-)
6. Ranking berdasarkan Ci terbesar

In [ ]:
def topsis_method(df, criteria, weights, benefit):
    """
    TOPSIS - Technique for Order Preference by Similarity to Ideal Solution
    Returns: DataFrame dengan skor TOPSIS dan ranking
    """
    X = df[criteria].values.astype(float)
    n, m = X.shape
    
    # Step 1: Normalisasi Vector
    denom = np.sqrt((X**2).sum(axis=0))
    denom = np.where(denom == 0, 1, denom)
    R = X / denom
    print('✅ Step 1: Normalisasi Vector selesai')
    
    # Step 2: Matriks Terbobot
    w_arr = np.array([weights[c] for c in criteria])
    V = R * w_arr
    print('✅ Step 2: Matriks Terbobot selesai')
    
    # Step 3: Solusi Ideal
    A_pos = np.zeros(m)
    A_neg = np.zeros(m)
    for j, crit in enumerate(criteria):
        if benefit[crit]:
            A_pos[j] = V[:, j].max()
            A_neg[j] = V[:, j].min()
        else:
            A_pos[j] = V[:, j].min()
            A_neg[j] = V[:, j].max()
    
    print('\n📍 Solusi Ideal:')
    for j, crit in enumerate(criteria):
        print(f'   {crit:<20} A+ = {A_pos[j]:.4f}  |  A- = {A_neg[j]:.4f}')
    
    # Step 4: Jarak Euclidean
    D_pos = np.sqrt(((V - A_pos)**2).sum(axis=1))
    D_neg = np.sqrt(((V - A_neg)**2).sum(axis=1))
    print('\n✅ Step 4: Jarak Euclidean selesai')
    
    # Step 5: Nilai Preferensi
    denom_ci = D_pos + D_neg
    denom_ci = np.where(denom_ci == 0, 1e-10, denom_ci)
    C = D_neg / denom_ci
    
    result = df[['Place_Name', 'City', 'Category'] + criteria].copy()
    result['D_pos']       = np.round(D_pos, 4)
    result['D_neg']       = np.round(D_neg, 4)
    result['TOPSIS_Score']= np.round(C, 4)
    result['TOPSIS_Rank'] = result['TOPSIS_Score'].rank(ascending=False, method='min').astype(int)
    return result.sort_values('TOPSIS_Rank').reset_index(drop=True), V, A_pos, A_neg

df_topsis, V_topsis, A_pos, A_neg = topsis_method(df_filtered, CRITERIA, WEIGHTS, BENEFIT)

print('\n🏆 TOP 10 Destinasi - Metode TOPSIS')
print('='*90)
display(df_topsis[['TOPSIS_Rank','Place_Name','City','Price','Rating','Jumlah_Review','Time_Minutes','D_pos','D_neg','TOPSIS_Score']].head(10))

## 7. Metode SMART (Simple Multi-Attribute Rating Technique)

**Langkah-langkah SMART:**
1. Normalisasi nilai tiap kriteria ke skala 0–100
2. Kalikan dengan bobot
3. Jumlahkan sebagai skor akhir

In [ ]:
def smart_method(df, criteria, weights, benefit):
    """
    SMART - Simple Multi-Attribute Rating Technique
    Normalisasi ke skala 0-100, lalu weighted sum
    """
    X = df[criteria].values.astype(float)
    U = np.zeros_like(X)  # Utility matrix (0-100)
    
    print('📊 Normalisasi SMART (skala 0-100):')
    for j, crit in enumerate(criteria):
        col = X[:, j]
        min_v, max_v = col.min(), col.max()
        rng = max_v - min_v
        if rng == 0:
            U[:, j] = 100.0
        elif benefit[crit]:
            U[:, j] = (col - min_v) / rng * 100
        else:  # Cost → invert
            U[:, j] = (max_v - col) / rng * 100
        print(f'   {crit:<20} min={min_v:.1f}  max={max_v:.1f}  type={"Benefit" if benefit[crit] else "Cost"}')
    
    # Weighted sum
    w_arr = np.array([weights[c] for c in criteria])
    scores = U.dot(w_arr)
    
    result = df[['Place_Name', 'City', 'Category'] + criteria].copy()
    for j, crit in enumerate(criteria):
        result[f'U_{crit}'] = np.round(U[:, j], 2)
    result['SMART_Score'] = np.round(scores, 4)
    result['SMART_Rank']  = result['SMART_Score'].rank(ascending=False, method='min').astype(int)
    return result.sort_values('SMART_Rank').reset_index(drop=True), U

df_smart, U_smart = smart_method(df_filtered, CRITERIA, WEIGHTS, BENEFIT)

print('\n🏆 TOP 10 Destinasi - Metode SMART')
print('='*80)
display(df_smart[['SMART_Rank','Place_Name','City','Price','Rating','Jumlah_Review','Time_Minutes','SMART_Score']].head(10))

## 8. Perbandingan & Analisis Ketiga Metode

In [ ]:
# Gabungkan hasil ketiga metode
df_compare = df_filtered[['Place_Name', 'City', 'Price', 'Rating', 'Jumlah_Review', 'Time_Minutes']].copy()

# Merge skor
df_compare = df_compare.merge(
    df_saw[['Place_Name','SAW_Score','SAW_Rank']], on='Place_Name', how='left')
df_compare = df_compare.merge(
    df_topsis[['Place_Name','TOPSIS_Score','TOPSIS_Rank']], on='Place_Name', how='left')
df_compare = df_compare.merge(
    df_smart[['Place_Name','SMART_Score','SMART_Rank']], on='Place_Name', how='left')

# Rank rata-rata (semakin kecil semakin baik)
df_compare['Avg_Rank'] = (df_compare['SAW_Rank'] + df_compare['TOPSIS_Rank'] + df_compare['SMART_Rank']) / 3
df_compare['Final_Rank'] = df_compare['Avg_Rank'].rank(method='min').astype(int)
df_compare = df_compare.sort_values('Final_Rank').reset_index(drop=True)

print('🏆 PERBANDINGAN RANKING TOP 15 DESTINASI WISATA')
print('='*100)
cols_show = ['Final_Rank','Place_Name','City','SAW_Rank','TOPSIS_Rank','SMART_Rank','Avg_Rank','SAW_Score','TOPSIS_Score','SMART_Score']
display(df_compare[cols_show].head(15))

In [ ]:
# Visualisasi perbandingan TOP 10
top10 = df_compare.head(10).copy()
top10['Label'] = top10['Place_Name'].str[:20]  # truncate

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Perbandingan Skor Top 10 Destinasi - Tiga Metode SPK', fontsize=14, fontweight='bold')

method_info = [
    ('SAW_Score',    'Metode SAW',    '#2196F3'),
    ('TOPSIS_Score', 'Metode TOPSIS', '#4CAF50'),
    ('SMART_Score',  'Metode SMART',  '#FF5722'),
]

for ax, (col, title, color) in zip(axes, method_info):
    bars = ax.barh(top10['Label'], top10[col], color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, top10[col]):
        ax.text(bar.get_width()*0.5, bar.get_y()+bar.get_height()/2,
                f'{val:.3f}', ha='center', va='center', color='white', fontweight='bold', fontsize=8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Skor')
    ax.invert_yaxis()
    ax.set_xlim(0, top10[col].max()*1.15)

plt.tight_layout()
plt.savefig('comparison_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap konsistensi ranking
top15 = df_compare.head(15)[['Place_Name','SAW_Rank','TOPSIS_Rank','SMART_Rank']].copy()
top15 = top15.set_index('Place_Name')
top15.index = top15.index.str[:20]

plt.figure(figsize=(8, 8))
sns.heatmap(top15, annot=True, fmt='.0f', cmap='YlOrRd_r',
            linewidths=0.5, cbar_kws={'label': 'Ranking (lebih kecil = lebih baik)'})
plt.title('Heatmap Ranking Top 15 Destinasi\n(3 Metode SPK)', fontweight='bold', pad=12)
plt.xlabel('Metode')
plt.ylabel('Destinasi Wisata')
plt.xticks(rotation=0)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('heatmap_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Korelasi Spearman antar metode
from scipy.stats import spearmanr

pairs = [('SAW_Rank','TOPSIS_Rank'), ('SAW_Rank','SMART_Rank'), ('TOPSIS_Rank','SMART_Rank')]
print('📊 KORELASI SPEARMAN ANTAR METODE')
print('='*45)
for a, b in pairs:
    r, p = spearmanr(df_compare[a], df_compare[b])
    status = '✅ Konsisten' if r > 0.8 else '⚠️ Perbedaan signifikan'
    print(f'  {a.split("_")[0]} vs {b.split("_")[0]}: r = {r:.4f}  p = {p:.4f}  {status}')

## 9. Analisis per Kota

In [ ]:
# Destinasi terbaik per kota (berdasarkan Final_Rank)
best_per_city = df_compare.groupby('City').first().reset_index()

print('🌟 DESTINASI TERBAIK PER KOTA (Final Ranking)')
print('='*80)
for _, row in best_per_city.iterrows():
    print(f"  {row['City']:<15} → {row['Place_Name']:<30} "
          f"SAW:{row['SAW_Score']:.3f}  TOPSIS:{row['TOPSIS_Score']:.3f}  SMART:{row['SMART_Score']:.3f}")

In [ ]:
# Visualisasi Top destinasi per kota
fig, axes = plt.subplots(1, len(best_per_city), figsize=(18, 6), sharey=False)
fig.suptitle('Skor TOPSIS Destinasi Wisata per Kota', fontsize=13, fontweight='bold')

palette = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6']
all_cities = df_compare['City'].unique()

for i, city in enumerate(all_cities):
    city_data = df_compare[df_compare['City']==city].head(5)
    ax = axes[i]
    bars = ax.barh(city_data['Place_Name'].str[:18], city_data['TOPSIS_Score'],
                   color=palette[i], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, city_data['TOPSIS_Score']):
        ax.text(bar.get_width()*0.5, bar.get_y()+bar.get_height()/2,
                f'{val:.3f}', ha='center', va='center', color='white', fontweight='bold', fontsize=8)
    ax.set_title(city, fontweight='bold', color=palette[i])
    ax.set_xlabel('TOPSIS Score')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('top_per_city.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Export Hasil ke CSV & Excel

In [ ]:
# Export ke CSV
df_compare.to_csv('hasil_spk_tourism.csv', index=False)
df_saw.to_csv('hasil_saw.csv', index=False)
df_topsis.to_csv('hasil_topsis.csv', index=False)
df_smart.to_csv('hasil_smart.csv', index=False)

# Export ke Excel (multi-sheet)
with pd.ExcelWriter('hasil_spk_lengkap.xlsx', engine='openpyxl') as writer:
    df_compare.to_excel(writer, sheet_name='Perbandingan', index=False)
    df_saw.to_excel(writer, sheet_name='SAW', index=False)
    df_topsis.to_excel(writer, sheet_name='TOPSIS', index=False)
    df_smart.to_excel(writer, sheet_name='SMART', index=False)
    df_filtered.to_excel(writer, sheet_name='Data Bersih', index=False)

print('✅ File berhasil diekspor:')
print('   📄 hasil_spk_tourism.csv')
print('   📄 hasil_saw.csv')
print('   📄 hasil_topsis.csv')
print('   📄 hasil_smart.csv')
print('   📊 hasil_spk_lengkap.xlsx (5 sheet)')

## 11. Ringkasan Hasil & Kesimpulan

In [ ]:
top5 = df_compare.head(5)

print('='*80)
print('  🏝️  KESIMPULAN SPK PEMILIHAN DESTINASI WISATA INDONESIA')
print('='*80)
print(f'\n  Dataset    : {len(df_filtered)} destinasi wisata')
print(f'  Kriteria   : {len(CRITERIA)} kriteria (Price, Rating, Jumlah Review, Waktu Kunjungan)')
print(f'  Metode     : SMART + SAW + TOPSIS')
print()
print('  TOP 5 DESTINASI WISATA TERBAIK (Gabungan 3 Metode):')
print('  ' + '-'*76)
for _, row in top5.iterrows():
    print(f"  #{int(row['Final_Rank']):<2} {row['Place_Name']:<28} [{row['City']:<15}] "
          f"Rating:{row['Rating']:.1f}  Harga:Rp{row['Price']:,.0f}  "
          f"TOPSIS:{row['TOPSIS_Score']:.3f}")
print('  ' + '-'*76)
print()
print('  Bobot Kriteria (SMART):')
for c in CRITERIA:
    tipe = 'Benefit↑' if BENEFIT[c] else 'Cost↓  '
    print(f'  • {CRITERIA_LABELS[c]:<30}: {WEIGHTS[c]:.0%} ({tipe})')
print('='*80)

---
## 🚀 Deployment
File Streamlit App tersedia di `streamlit_app.py`.

Jalankan dengan:
```bash
streamlit run streamlit_app.py
```